# Master — REGISTRACIJA

Cilj je dokazati brojčano koliko registracija doprinosi.



## Ćelija 0 — priprema

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q SimpleITK nibabel

import os, glob, tarfile, shutil
import numpy as np
import SimpleITK as sitk

MASTER = '/content/drive/MyDrive/Master/kicma'
print('spremno')

Mounted at /content/drive
spremno


## Ćelija 1

In [ ]:
ROOT = '/content/msspine'
if not os.path.isdir(f'{ROOT}/rawdata'):
    tar = None
    for h in glob.glob('/content/drive/MyDrive/**/MS_MultiSpine_dataset*.tar.gz', recursive=True):
        tar = h; break
    raw = None
    for h in glob.glob('/content/drive/MyDrive/**/rawdata', recursive=True):
        if os.path.isdir(h): raw = h; break
    if raw:
        ROOT = os.path.dirname(raw)
        print('nadjen raspakovan rawdata:', ROOT)
    elif tar:
        os.makedirs(ROOT, exist_ok=True)
        print('raspakujem', tar)
        with tarfile.open(tar) as t: t.extractall(ROOT)
        for h in glob.glob(f'{ROOT}/**/rawdata', recursive=True):
            ROOT = os.path.dirname(h); break
    else:
        raise FileNotFoundError('Ni .tar.gz ni rawdata nisu nadjeni na Drive-u.')
RAW = f'{ROOT}/rawdata'
print('rawdata:', RAW, '| pacijenata:', len(glob.glob(f"{RAW}/sub-*")))

raspakujem /content/drive/MyDrive/Master/kicma/podaci_sirovi/MS_MultiSpine_dataset.tar.gz


/tmp/ipykernel_603/2461370799.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar) as t: t.extractall(ROOT)


rawdata: /content/msspine/MS_MultiSpine_dataset/rawdata | pacijenata: 100


## Ćelija 2 — mapiranje sekvenci po pacijentu

In [ ]:
def nadji(sub_dir, tipovi):
    for t in tipovi:
        hits = glob.glob(f'{sub_dir}/*_{t}.nii.gz')
        if hits: return hits[0]
    return None

pacijenti = []
for sub in sorted(glob.glob(f'{RAW}/sub-*')):
    t2  = nadji(sub, ['T2'])
    add = nadji(sub, ['STIR','PSIR','MP2RAGE','UNIT1'])
    if t2 and add:
        seq = os.path.basename(add).split('_')[-1].replace('.nii.gz','')
        pacijenti.append((os.path.basename(sub), t2, add, seq))
from collections import Counter
print('parova (T2 + dodatna):', len(pacijenti))
print('po sekvenci:', Counter(s for *_,s in pacijenti))

parova (T2 + dodatna): 100
po sekvenci: Counter({'STIR': 50, 'PSIR': 25, 'MP2RAGE': 25})


## Ćelija 3 — registracija + merenje (PRE vs POSLE)
Za svakog: MI i NCC između T2 i dodatne, (a) samo preuzorkovana, (b) registrovana.
MI (Mattes) — niže je bolje poravnato; NCC — bliže 0 (negativnije) bolje.

In [ ]:
import time
def metrika_mi(fixed, moving):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(0.1, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetInitialTransform(sitk.Transform(fixed.GetDimension(), sitk.sitkIdentity))
    return R.MetricEvaluate(fixed, moving)

def registruj(fixed, moving):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(0.1, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetOptimizerAsRegularStepGradientDescent(learningRate=0.3, minStep=1e-4,
                                               numberOfIterations=80,
                                               gradientMagnitudeTolerance=1e-6,
                                               relaxationFactor=0.5)
    R.SetOptimizerScalesFromPhysicalShift()
    R.SetShrinkFactorsPerLevel([4,2]); R.SetSmoothingSigmasPerLevel([2,1])
    R.SetInitialTransform(sitk.Euler3DTransform(), inPlace=False)
    return R.Execute(fixed, moving)

SUBSET = None
skup = pacijenti if SUBSET is None else pacijenti[:SUBSET]

SAVE = f'{MASTER}/registrovano'; os.makedirs(SAVE, exist_ok=True)
rezultati = []
t0 = time.time()
for i,(pid, t2p, addp, seq) in enumerate(skup, 1):
    tp = time.time()
    try:
        fixed  = sitk.ReadImage(t2p,  sitk.sitkFloat32)
        moving = sitk.ReadImage(addp, sitk.sitkFloat32)
        naive = sitk.Resample(moving, fixed, sitk.Transform(3, sitk.sitkIdentity),
                              sitk.sitkLinear, 0.0, moving.GetPixelID())
        mi_pre = metrika_mi(fixed, naive)
        T = registruj(fixed, moving)
        reg = sitk.Resample(moving, fixed, T, sitk.sitkLinear, 0.0, moving.GetPixelID())
        mi_post = metrika_mi(fixed, reg)
        if mi_post > mi_pre:
            reg = naive; mi_post = mi_pre; pomereno = False
        else:
            pomereno = True
        sitk.WriteImage(reg, f'{SAVE}/{pid}_{seq}_reg.nii.gz')
        rezultati.append((pid, seq, mi_pre, mi_post, pomereno))
        print(f'[{i}/{len(skup)}] {pid} {seq}: pre={mi_pre:.3f} posle={mi_post:.3f} '
              f'{"OK" if pomereno else "zadrzan naive"}  ({time.time()-tp:.1f}s)')
    except Exception as e:
        print(f'[{i}/{len(skup)}] greska {pid}:', e)
print(f'\ngotovo: {len(rezultati)} za {(time.time()-t0)/60:.1f} min | registracija pomerila: '
      f'{sum(1 for *_,m in rezultati if m)}')

[1/100] sub-001 STIR: pre=-0.667 posle=-0.678 OK  (11.6s)
[2/100] sub-002 STIR: pre=-0.467 posle=-0.481 OK  (9.1s)
[3/100] sub-003 STIR: pre=-0.432 posle=-0.452 OK  (30.0s)
[4/100] sub-004 STIR: pre=-0.345 posle=-0.345 zadrzan naive  (10.2s)
[5/100] sub-005 STIR: pre=-0.443 posle=-0.503 OK  (7.8s)
[6/100] sub-006 STIR: pre=-0.604 posle=-0.626 OK  (7.7s)
[7/100] sub-007 STIR: pre=-0.708 posle=-0.708 zadrzan naive  (23.4s)
[8/100] sub-008 STIR: pre=-0.385 posle=-0.399 OK  (23.6s)
[9/100] sub-009 STIR: pre=-0.732 posle=-0.741 OK  (12.3s)
[10/100] sub-010 STIR: pre=-0.466 posle=-0.466 zadrzan naive  (8.4s)
[11/100] sub-011 STIR: pre=-0.592 posle=-0.592 zadrzan naive  (18.7s)
[12/100] sub-012 STIR: pre=-0.257 posle=-0.257 zadrzan naive  (15.1s)
[13/100] sub-013 STIR: pre=-0.442 posle=-0.502 OK  (27.2s)
[14/100] sub-014 STIR: pre=-0.263 posle=-0.263 zadrzan naive  (7.4s)
[15/100] sub-015 STIR: pre=-0.526 posle=-0.536 OK  (27.1s)
[16/100] sub-016 STIR: pre=-0.398 posle=-0.433 OK  (8.7s)
[17/1

## Ćelija 4 — poboljšanje po sekvenci

In [ ]:
import numpy as np
from collections import defaultdict
po_seq = defaultdict(list)
for pid, seq, pre, post, mov in rezultati:
    po_seq[seq].append(post - pre)

print(f"{'sekvenca':10} {'N':>4} {'MI pre':>9} {'MI posle':>9} {'Δ (bolje<0)':>12} {'% boljih':>9}")
svi_pre=[p for _,_,p,_,_ in rezultati]; svi_post=[p for _,_,_,p,_ in rezultati]
for seq in sorted(po_seq):
    ds = po_seq[seq]
    pre  = np.mean([p for pid,s,p,_,_ in rezultati if s==seq])
    post = np.mean([p for pid,s,_,p,_ in rezultati if s==seq])
    pobo = 100*np.mean([d<0 for d in ds])
    print(f"{seq:10} {len(ds):>4} {pre:>9.4f} {post:>9.4f} {np.mean(ds):>12.4f} {pobo:>8.0f}%")
print('-'*58)
print(f"{'UKUPNO':10} {len(rezultati):>4} {np.mean(svi_pre):>9.4f} {np.mean(svi_post):>9.4f} "
      f"{np.mean(svi_post)-np.mean(svi_pre):>12.4f} {100*np.mean([b-a<0 for a,b in zip(svi_pre,svi_post)]):>8.0f}%")
print('\nNiza Mattes MI = bolja poravnatost. Δ<0 = registracija poboljsala.')
print('(Sa zastitom: registracija nikad ne vraca losije od pocetne poravnatosti.)')

import csv
rd = f'{MASTER}/rezultati'; os.makedirs(rd, exist_ok=True)
with open(f'{rd}/registracija.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['pacijent','sekvenca','MI_pre','MI_posle','pomereno'])
    for r in rezultati: w.writerow(r)
print('tabela:', f'{rd}/registracija.csv')

sekvenca      N    MI pre  MI posle  Δ (bolje<0)  % boljih
MP2RAGE      25   -0.1937   -0.2221      -0.0284      100%
PSIR         25   -0.2332   -0.2446      -0.0114       72%
STIR         50   -0.4524   -0.4663      -0.0139       58%
----------------------------------------------------------
UKUPNO      100   -0.3329   -0.3498      -0.0169       72%

Niza Mattes MI = bolja poravnatost. Δ<0 = registracija poboljsala.
(Sa zastitom: registracija nikad ne vraca losije od pocetne poravnatosti.)
tabela: /content/drive/MyDrive/Master/kicma/rezultati/registracija.csv
